# Real-Time Building Energy Prediction with Spark Structured Streaming


## Overview

This notebook implements the **real-time streaming inference pipeline**. It subscribes to a Kafka topic populated by the weather data producer (notebook 02), enriches each record with building metadata, applies the pre-trained GBT pipeline from notebook 01, and persists predictions in three aggregation formats.

### Pipeline Position

```
Kafka Topic: weather-stream --> [03 Spark Streaming] --> output/predictions
                                         ^               output/hourly_aggregations
                             models/energy_pipeline_model  output/daily_aggregations
                             (trained in Notebook 01)
```

### Dependencies

| Dependency | Required state |
|------------|----------------|
| **Notebook 01** | `models/energy_pipeline_model` must exist |
| **Notebook 02** | Must be actively publishing to `weather-stream` |
| **Docker** | Kafka broker running at `kafka:9092` |

### Outputs

| Path | Description |
|------|-------------|
| `output/predictions` | Raw per-record predictions (Parquet, append) |
| `output/hourly_aggregations` | Total energy per building per 6-hour window |
| `output/daily_aggregations` | Total energy per site per day |


In [ ]:
import os
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from pyspark import SparkContext
from pyspark import SparkConf
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.ml.feature import Imputer
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pyspark.ml.feature import OneHotEncoder, StringIndexer
import math
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import PipelineModel
from pyspark.sql.functions import to_json, struct

## 1. Spark Session Setup

Initialise a local SparkSession with 4 cores and the Kafka integration packages loaded via `PYSPARK_SUBMIT_ARGS`. The session timezone is set to Melbourne to align event-time windows with the source data, and a checkpoint base path is defined for all streaming queries.


In [ ]:
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.4.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0 pyspark-shell'

# Create Spark session
app_name = "building-energy-streaming"
checkpoint = "checkpoint/streaming"
spark = (
    SparkSession.builder.appName(app_name)
    .master("local[4]") # 4 cores
    .config("spark.sql.session.timeZone", "Australia/Melbourne") # Melbourne time zone
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

#get sparkcontext from the sparksession
sc = spark.sparkContext

## 2. Schema Definitions and Static Data

Define typed schemas for:
- **`buildings_schema`** - static building attributes loaded from CSV
- **`weather_row_schema`** - a single weather record as received from the Kafka producer
- **`weather_schema`** - the outer Kafka message envelope (`ts` + array of weather rows)

The building DataFrame is loaded once as a static lookup table and joined into the stream in the next section.


In [ ]:
# buildings.csv
buildings_schema = StructType([
    StructField("site_id", IntegerType(), True),
    StructField("building_id", IntegerType(), True),
    StructField("primary_use", StringType(), True),
    StructField("square_feet", IntegerType(), True),
    StructField("floor_count", IntegerType(), True),
    StructField("row_id", IntegerType(), True),
    StructField("year_built", IntegerType(), True),
    StructField("latent_y", DoubleType(), True),
    StructField("latent_s", DoubleType(), True),
    StructField("latent_r", DoubleType(), True)
])

In [ ]:
# weather.csv
weather_row_schema = StructType([
    StructField("site_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("air_temperature", StringType(), True),
    StructField("cloud_coverage", StringType(), True),
    StructField("dew_temperature", StringType(), True),
    StructField("sea_level_pressure", StringType(), True),
    StructField("wind_direction", StringType(), True),
    StructField("wind_speed", StringType(), True),
    StructField("weather_ts", StringType(), True)
])

weather_schema = StructType([
    StructField("ts", StringType(), True),
    StructField("rows", ArrayType(weather_row_schema), True)
])

In [ ]:
# json file from kafka producer
kafka_schema = StructType([
    StructField("ts", StringType(), True),
    StructField("rows", ArrayType(weather_schema), True)
])

In [ ]:
# buildings
buildings_df = spark.read.csv("data/new_building_information.csv",header=True,schema=buildings_schema)
print("Buildings DataFrame Schema:")
buildings_df.printSchema()

## 3. Kafka Stream Ingestion

Subscribe to the `weather-stream` Kafka topic and parse the incoming JSON payload into the defined schema. String fields are cast to their correct numeric types, and the stream is joined with the static building DataFrame on `site_id`.


In [ ]:
# configuration
hostip = "kafka"
topic = 'weather-stream'
# load data from Kafka topic
df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", f"{hostip}:9092") \
    .option("subscribe", topic) \
    .load()
# converting the key/value to string
df = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")
# transformed into the metadata file schema
df = df.select(F.from_json(F.col("value"), weather_schema).alias("data"))
df = df.selectExpr("data.ts as batch_ts", "explode(data.rows) as row")
df = df.select("batch_ts", "row.*")

# transformed into the proper formats
numeric_cols_double = [
    "air_temperature", "dew_temperature",
    "sea_level_pressure", "wind_speed"
]

numeric_cols_int = ["site_id", "cloud_coverage", "wind_direction"]

for col_name in numeric_cols_double:
    df = df.withColumn(
        col_name,
        F.when(F.col(col_name) == "", None)
         .otherwise(F.col(col_name).cast("double"))
    )

for col_name in numeric_cols_int:
    df = df.withColumn(
        col_name,
        F.when(F.col(col_name) == "", None)
         .otherwise(F.col(col_name).cast("int"))
    )

In [ ]:
# join building dataframe
feature_df = (df.join(buildings_df, on="site_id", how="left"))

## 4. Event Time and Watermarking

Convert the `weather_ts` Unix timestamp (carried in the Kafka payload) to a Spark `TimestampType` column named `event_time`. A 5-second watermark is applied so that Spark can safely clean up state: any record arriving more than 5 seconds after its event time is discarded.


In [ ]:
# create timestamp column from weather_ts
feature_df = feature_df.withColumn("event_time",F.to_timestamp(F.col("weather_ts").cast("double")))

# watermark for 5 sec
feature_df = feature_df.withWatermark("event_time", "5 seconds")

## 5. Feature Engineering

Apply the same transformations used to train the batch ML model in notebook 01, so the streaming DataFrame schema matches the pipeline's expected feature vector:

- **Log transform** `square_feet` -> `log_square_feet`
- **Cyclical encoding** of `time_interval`, `wind_direction`, and `month` (sine + cosine)
- **Decade** derived from `year_built`
- **Zero-fill** any residual nulls introduced by the stream join


In [ ]:
# reuse code from notebook 01
feature_df = (feature_df.withColumn("year", F.year("event_time"))
                        .withColumn("month", F.month("event_time"))
                        .withColumn("day", F.dayofmonth("event_time"))
                        .withColumn("time_interval", F.hour("event_time")))

# cyclical encoding
# time_interval
# transfer to numerical 
feature_df = feature_df.withColumn(
    "time_interval_num",
    F.when(F.col("time_interval") == "0:00-5:59", 0)
     .when(F.col("time_interval") == "6:00-11:59", 1)
     .when(F.col("time_interval") == "12:00-17:59", 2)
     .when(F.col("time_interval") == "18:00-23:59", 3)
     .otherwise(None)
)

feature_df = feature_df.withColumn("time_sin",F.sin(2 * math.pi * F.col("time_interval_num") / 4))
feature_df = feature_df.withColumn("time_cos", F.cos(2 * math.pi * F.col("time_interval_num") / 4))

# wind_direction
feature_df = feature_df.withColumn("wind_dir_sin", F.sin(2 * math.pi * F.col("wind_direction")/360))
feature_df = feature_df.withColumn("wind_dir_cos", F.cos(2 * math.pi * F.col("wind_direction")/360))
# date features
# drop year (only 2022)
feature_df = feature_df.drop("year")

# month cyclical encoding
feature_df = feature_df.withColumn("month_sin", F.sin(2 * math.pi * F.col("month")/12))
feature_df = feature_df.withColumn("month_cos", F.cos(2 * math.pi * F.col("month")/12))

# drop day
feature_df = feature_df.drop("day")

# group year_built by decades
feature_df = feature_df.withColumn("decade", (F.col("year_built")/10).cast("int")*10)
feature_df = feature_df.withColumn("log_square_feet", F.log1p(F.col("square_feet")))

cols_to_fill = [
    "building_id","site_id","wind_direction","decade",
    "air_temperature","cloud_coverage","dew_temperature","sea_level_pressure",
    "wind_speed","log_square_feet","floor_count","latent_y","latent_s","latent_r",
    "time_sin","time_cos","wind_dir_sin","wind_dir_cos","month_sin","month_cos"
]

# fill missing value for safety
feature_df = feature_df.fillna(0, subset=cols_to_fill)

## 6. Model Inference and Streaming Queries

Load the saved GBT pipeline and apply it to the live feature stream. Three queries demonstrate different output modes and trigger intervals.


In [ ]:
# apply the saved GBT pipeline
pipeline = PipelineModel.load("models/energy_pipeline_model")
pred_stream = pipeline.transform(feature_df)

### 6a. Per-Record Predictions (Memory Sink)

Write raw predictions to an in-memory table (`pred_table`) for immediate inspection. Triggered every 5 seconds in append mode.


In [ ]:
# create a streaming query to write prediction results into a memory table
query_pred = (pred_stream
         .select("event_time", "site_id", "building_id", "prediction") # select the needed column
         .writeStream
         .outputMode("append")
         .format("memory")
         .queryName("pred_table")
         .trigger(processingTime="5 seconds")
         .option("checkpointLocation", f"{checkpoint}/parquet_pred") 
         .start())

In [ ]:
# check the result
spark.sql("SELECT * FROM pred_table").show(20, truncate=False)

In [ ]:
# stop the query
query_pred.stop()

### 6b. 6-Hour Window Aggregation per Building

Aggregate total predicted energy consumption over tumbling 6-hour windows, grouped by `building_id`. Uses `complete` output mode and triggers every 7 seconds.


In [ ]:
# 6b
# 6 hour window aggregation
energy_window_df = (
    pred_stream
    .groupBy(F.window(F.col("event_time"), "6 hours"), F.col("building_id"))
    .agg(F.sum("prediction").alias("total_energy"))
)

In [ ]:
# create the query to see the total energy for each building
query_mem_6h = (
    energy_window_df
    .writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("energy_table")
    .option("checkpointLocation", f"{checkpoint}/mem_6h")
    .trigger(processingTime="7 seconds")
    .start()
)

In [ ]:
# check the result
spark.sql("SELECT * FROM energy_table").show(20, truncate=False)

In [ ]:
# stop the query
query_mem_6h.stop()

### 6c. Daily Aggregation per Site

Aggregate total predicted energy consumption over tumbling 1-day windows, grouped by `site_id`. Uses `complete` output mode and triggers every 14 seconds.


In [ ]:
daily_energy_df = (
    pred_stream
    .groupBy( F.window(F.col("event_time"), "1 day"), F.col("site_id"))
    .agg(F.sum("prediction").alias("daily_total_energy"))
)

In [ ]:
# create the query to see the daily energy consumption for each site
query_mem_daily = (
    daily_energy_df
    .writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("daily_table")
    .option("checkpointLocation", f"{checkpoint}/mem_daily")
    .trigger(processingTime="14 seconds")
    .start()
)

In [ ]:
# check the result
spark.sql("SELECT * FROM daily_table").show(truncate=False)

In [ ]:
# stop the query
query_mem_daily.stop()

## 7. Persist Streams to Parquet

Write each of the three result streams to Parquet on disk. Files are updated incrementally with each micro-batch, making them available for downstream batch reads or further streaming queries.


### 7a. Raw Predictions

Append-mode stream writing raw predictions to `output/predictions`. Triggered every 5 seconds.


In [ ]:
parquet_query_predictions = (
    pred_stream.select("event_time", "site_id", "building_id", "prediction").writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", "output/predictions")
    .option("checkpointLocation", f"{checkpoint}/predictions_stream")
    .trigger(processingTime="5 seconds")
    .start()
)

In [ ]:
df_predictions = spark.read.parquet("output/predictions")
df_predictions.show(5, truncate=False)

In [ ]:
# stop the query
parquet_query_predictions.stop()

### 7b. 6-Hour Aggregations

Uses `foreachBatch` to overwrite `output/hourly_aggregations` with the latest complete window state. Triggered every 7 seconds.


In [ ]:
def write_to_parquet_hourly(batch_df, batch_id):
    (batch_df
        .write
        .mode("overwrite")
        .parquet(f"output/hourly_aggregations")
    )

parquet_query_hourly = (
    energy_window_df.writeStream
    .foreachBatch(write_to_parquet_hourly)
    .outputMode("complete")
    .trigger(processingTime="7 seconds")
    .option("checkpointLocation", f"{checkpoint}/hourly")
    .start()
)

In [ ]:
df_hourly = spark.read.parquet("output/hourly_aggregations")
df_hourly.show(5, truncate=False)

In [ ]:
# stop the query
parquet_query_hourly.stop()

### 7c. Daily Aggregations

Uses `foreachBatch` to overwrite `output/daily_aggregations` with the latest complete window state. Triggered every 14 seconds.


In [ ]:
def write_to_parquet_daily(batch_df, batch_id):
    (batch_df
        .write
        .mode("overwrite")
        .parquet(f"output/daily_aggregations")
    )

parquet_query_daily = (
    daily_energy_df.writeStream
    .foreachBatch(write_to_parquet_daily)
    .outputMode("complete")
    .trigger(processingTime="14 seconds")
    .option("checkpointLocation", f"{checkpoint}/daily")
    .start()
)

In [ ]:
df_daily= spark.read.parquet("output/daily_aggregations")
df_daily.show(truncate=False)

In [ ]:
# stop the query
parquet_query_daily.stop()

## 8. Re-publish Results to Kafka

Read each Parquet output directory as a streaming source and forward records to a dedicated Kafka topic as JSON, making results available to any downstream consumer.

| Parquet source | Kafka topic |
|----------------|-------------|
| `output/predictions` | `predictions-stream` |
| `output/hourly_aggregations` | `hourly-stream` |
| `output/daily_aggregations` | `daily-stream` |


### 8a. Raw Predictions -> `predictions-stream`


In [ ]:
# Stream 1

#  Read Parquet files as streaming source
parquet_stream_8a = (
    spark.readStream
    .schema(pred_stream.schema)
    .option("maxFilesPerTrigger", 1)
    .parquet("output/predictions")
)

# Convert to JSON for Kafka
kafka_ready_8a = parquet_stream_8a.selectExpr("CAST(null AS STRING) AS key","to_json(struct(*)) AS value")

# Send to Kafka
kafka_query_8a = (
    kafka_ready_8a.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", "predictions-stream")
    .option("checkpointLocation", f"{checkpoint}/kafka_8a")
    .outputMode("append")
    .start()
)

In [ ]:
# stop the query
kafka_query_8a.stop()

### 8b. 6-Hour Aggregations -> `hourly-stream`


In [ ]:
# Stream 2

#  Read Parquet files as streaming source
parquet_stream_8b = (
    spark.readStream
    .schema(pred_stream.schema)
    .option("maxFilesPerTrigger", 1)
    .parquet("output/hourly_aggregations")
)

# Convert to JSON for Kafka
kafka_ready_8b = parquet_stream_8b.selectExpr(
    "CAST(null AS STRING) AS key",
    "to_json(struct(*)) AS value"
)

# Send to Kafka
kafka_query_8b = (
    kafka_ready_8b.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", "hourly-stream")
    .option("checkpointLocation", f"{checkpoint}/kafka_8b")
    .outputMode("append")
    .start()
)

In [ ]:
# stop the query
kafka_query_8b.stop()

### 8c. Daily Aggregations -> `daily-stream`


In [ ]:
# Stream 3

#  Read Parquet files as streaming source
parquet_stream_8c = (
    spark.readStream
    .schema(pred_stream.schema)
    .option("maxFilesPerTrigger", 1)
    .parquet("output/daily_aggregations")
)

# Convert to JSON for Kafka
kafka_ready_8c = parquet_stream_8c.selectExpr(
    "CAST(null AS STRING) AS key",
    "to_json(struct(*)) AS value"
)

# Send to Kafka
kafka_query_8c = (
    kafka_ready_8c.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", "daily-stream")
    .option("checkpointLocation", f"{checkpoint}/kafka_8c")
    .outputMode("append")
    .start()
)

In [ ]:
# stop the query
kafka_query_8c.stop()